# Why DeepSeek uses Mixture of Experts

This notebook studies expert routing and total versus active parameters. It does not implement the MoE layer yet.

In [1]:
import sys
from pathlib import Path

import torch

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "model").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the repository root")
sys.path.insert(0, str(project_root))

In [2]:
from src.model import DeepSeekConfig

config = DeepSeekConfig()
print("Routed experts:", config.n_routed_experts)
print("Shared experts:", config.n_shared_experts)
print("Selected routed experts:", config.n_activated_experts)
print("Expert intermediate dim:", config.moe_inter_dim)
print("Initial dense layers:", config.n_dense_layers)

Routed experts: 8
Shared experts: 1
Selected routed experts: 2
Expert intermediate dim: 256
Initial dense layers: 1


In [3]:
D = config.dim
I = config.moe_inter_dim
experts = config.n_routed_experts
active = config.n_activated_experts
shared = config.n_shared_experts

expert_params = 3 * D * I
router_params = D * experts
total = (experts + shared) * expert_params + router_params
active_path = (active + shared) * expert_params + router_params
dense_ffn = 3 * D * config.inter_dim

print(f"One expert: {expert_params:,}")
print(f"Total MoE sublayer: {total:,}")
print(f"Active path: {active_path:,}")
print(f"Dense FFN: {dense_ffn:,}")

One expert: 196,608
Total MoE sublayer: 1,771,520
Active path: 591,872
Dense FFN: 589,824


In [4]:
torch.manual_seed(15)
router = torch.sigmoid(torch.randn(1, config.n_routed_experts))
scores, expert_ids = torch.topk(router, config.n_activated_experts, dim=-1)
weights = scores / scores.sum(dim=-1, keepdim=True)

print("Router scores:", router.round(decimals=3))
print("Selected expert IDs:", expert_ids.tolist())
print("Normalized selected weights:", weights.round(decimals=3))
print("Expert FFNs on the token path:", config.n_activated_experts + config.n_shared_experts)

Router scores: tensor([[0.3310, 0.6620, 0.3670, 0.7130, 0.7440, 0.5360, 0.2180, 0.4560]])
Selected expert IDs: [[4, 3]]
Normalized selected weights: tensor([[0.5110, 0.4890]])
Expert FFNs on the token path: 3


The router experiment is illustrative. The expert networks and the actual routed computation will be added next.